In [1]:
import os

In [2]:
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/lispawel/data_project_n1.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "lispawel"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "3c7f7df8b33b6770b9e1e54bdde749818852a735"

In [3]:
os.chdir("../")
%pwd

'c:\\Users\\lispa\\Desktop\\Data Science\\MLOps\\data_project_n1'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [5]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories, save_json

In [6]:
class ConfigurationManager:
    def __init__(
        self, 
        config_file_path=CONFIG_FILE_PATH, 
        params_file_path=PARAMS_FILE_PATH,
        schema_file_path=SCHEMA_FILE_PATH
        ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.shcema = read_yaml(schema_file_path)
        
        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.shcema.TARGET_COLUMN.name

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir= config.root_dir,
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            all_params = params,
            metric_file_name = config.metric_file_name,
            target_column = schema,
            mlflow_uri = "https://dagshub.com/lispawel/data_project_n1.mlflow"
        )

        return model_evaluation_config

In [7]:
import os 
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

c:\Users\lispa\Desktop\Data Science\MLOps\data_project_n1\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            scores = {
                "rmse": rmse,
                "mae": mae,
                "r2": r2
            }
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)

            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, name="model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, name="model")
            

In [9]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise e

[2026-09-13 18:32:16,357:  INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-13 18:32:16,360:  INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-13 18:32:16,364:  INFO: common: yaml file: schema.yaml loaded successfully]
[2026-09-13 18:32:16,365:  INFO: common: created directory at: artifacts]
[2026-09-13 18:32:16,366:  INFO: common: created directory at: artifacts/model_evaluation]
[2026-09-13 18:32:28,526:  INFO: common: json file saved at: artifacts\model_evaluation\metric.json]


Successfully registered model 'ElasticnetModel'.
2026/09/13 18:32:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 1
Created version '1' of model 'ElasticnetModel'.


🏃 View run charming-whale-989 at: https://dagshub.com/lispawel/data_project_n1.mlflow/#/experiments/0/runs/ab8ff0e347224cd9bbe8073187731e50
🧪 View experiment at: https://dagshub.com/lispawel/data_project_n1.mlflow/#/experiments/0
